# Remote Distill Training

This notebook starts (or re-attaches to) an autonomous remote training job via `SlopClient`.
The job keeps running even if your laptop disconnects.


In [1]:
import sys
from pathlib import Path

root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root.resolve()))

PROVIDER_NAME = 'vast-32582479'
REMOTE_MANIFEST_PATH = '/root/slop/distill/data/test/grok_test/manifest.jsonl'
REMOTE_OUTPUT_DIR = '/root/slop/distill/output/grok_test'

MODEL_ID = 'runwayml/stable-diffusion-v1-5'
BATCH_SIZE = 2
EPOCHS = 5
LEARNING_RATE = 5e-5
SAVE_EVERY = 50

TIMEOUT_S = 7200
PROCEED = False


In [2]:
from client.config import registry
from client.interface import SlopClient

registry.refresh()
provider = registry.get(PROVIDER_NAME)
if provider is None:
    names = [p.name for p in registry.list()]
    raise RuntimeError(f'Provider not found: {PROVIDER_NAME}. Available: {names}')

with SlopClient(provider) as client:
    info = client.info()
    print(f'Server: {info.hostname} | GPU: {info.gpu_name} | VRAM: {info.gpu_memory_mb}MB')

    stats = client.dataset_stats(REMOTE_MANIFEST_PATH, sample_images=16)
    if stats.get('error'):
        raise RuntimeError(stats['error'])

    print('--- Dataset ---')
    print('manifest:', stats.get('manifest_path'))
    print('records_read:', stats.get('records_read'))
    print('images_ok:', stats.get('images_ok'), 'images_missing:', stats.get('images_missing'))
    print('unique_prompts:', stats.get('unique_prompts'))
    print('teachers:', stats.get('teachers'))
    bt = int(stats.get('bytes_total') or 0)
    print('bytes_total:', bt, f'(~{bt/1e6:.2f} MB)')
    print('sample_image_sizes:', stats.get('sample_image_sizes'))
    print('sample_prompts:')
    for p in stats.get('sample_prompts', []):
        print(' -', p)

    if int(stats.get('images_ok') or 0) == 0:
        print('WARNING: dataset has 0 readable images. Training will do nothing.')


[REMOTE STDERR] [SERVER] 2026-03-09 20:18:25,493 - INFO - server ready


Server: 6d4c5ec935f6 | GPU: NVIDIA GeForce RTX 3090 | VRAM: 24122MB
--- Dataset ---
manifest: /root/slop/distill/data/test/grok_test/manifest.jsonl
records_read: 1
images_ok: 0 images_missing: 1
unique_prompts: 1
teachers: {'openrouter': 1}
bytes_total: 0 (~0.00 MB)
sample_image_sizes: []
sample_prompts:
 - Portrait of an Arab man standing in a street market


In [3]:
from client.config import registry
from client.interface import SlopClient

registry.refresh()
provider = registry.get(PROVIDER_NAME)
if provider is None:
    names = [p.name for p in registry.list()]
    raise RuntimeError(f'Provider not found: {PROVIDER_NAME}. Available: {names}')

with SlopClient(provider) as client:
    info = client.info()
    print(f'Server: {info.hostname} | GPU: {info.gpu_name} | VRAM: {info.gpu_memory_mb}MB')

    if not PROCEED:
        print('Set PROCEED = True to start/attach training')
    else:
        running_states = {'starting','running','stopping'}
        jobs = client.list_jobs(limit=200)
        matches = [j for j in jobs if j.get('kind')=='train' and j.get('output_dir')==REMOTE_OUTPUT_DIR and j.get('state') in running_states]

        if matches:
            matches.sort(key=lambda j: float(j.get('updated_at') or 0.0), reverse=True)
            job_id = matches[0]['job_id']
            print('Re-attaching to running job:', job_id)
        else:
            start = client.train(
                manifest_path=REMOTE_MANIFEST_PATH,
                output_dir=REMOTE_OUTPUT_DIR,
                model_id=MODEL_ID,
                batch_size=BATCH_SIZE,
                epochs=EPOCHS,
                learning_rate=LEARNING_RATE,
                save_every=SAVE_EVERY,
                timeout_s=60.0,
            )
            job_id = start.metadata['job_id']
            print('Started autonomous job:', job_id)

        import time
        since = 0
        deadline = time.time() + TIMEOUT_S
        while time.time() < deadline:
            state = client.attach_job(job_id, since_line=since, max_lines=200)
            status = state.get('status', {})
            for ev in state.get('progress', []):
                try:
                    loss = float(ev.get('loss', 0.0))
                except Exception:
                    loss = 0.0
                print(f"[progress] ep={ev.get('epoch')} step={ev.get('step')} loss={loss:.4f} {ev.get('message','')}")
            since = int(state.get('next_since_line', since))
            st = status.get('state')
            if st in {'completed','failed','killed'}:
                print('Final status:', status)
                if st != 'completed':
                    print('Log tail:\n', state.get('log_tail',''))
                break
            time.sleep(2)
        else:
            print('Timed out; re-run this cell to re-attach (job keeps running).')


[REMOTE STDERR] [SERVER] 2026-03-09 20:18:32,988 - INFO - server ready


Server: 6d4c5ec935f6 | GPU: NVIDIA GeForce RTX 3090 | VRAM: 24122MB
Set PROCEED = True to start/attach training
